# Assignment 1:
### - The automated_stat_analyzer Function
- Scenario: A retail company needs a utility to quickly summarize sales data. Students must create a function that identifies the 
"Central Tendency" and "Dispersion" of any numerical column.
- ### Requirements:

* Accept a Pandas DataFrame and a column name.

* Calculate the Mean, Median, and Standard Deviation .

* Identify if the data is "Skewed" by comparing the Mean and Median.


* Bonus: If the column is categorical, return the Mode instead.

### Your Data

In [4]:
import pandas as pd
import numpy as np

# Create a synthetic Company Sales Dataset
data = {
    'Transaction_ID': range(1, 11),
    'Product_Category': ['Electronics', 'Home', 'Electronics', 'Sports', 'Home', 
                         'Electronics', 'Home', 'Sports', 'Electronics', 'Electronics'],
    'Sales_Amount': [150, 200, 155, 300, 210, 180, 205, 1000, 190, 160], # 1000 is an Outlier
    'Customer_Age': [25, 34, np.nan, 45, 23, 31, 29, np.nan, 38, 40],    # Contains Nulls (NaN)
    'Rating': [5, 4, 3, 5, 2, 4, 5, 2, 4, 3]
}

df_test = pd.DataFrame(data)

# Save to CSV for students to practice loading files [cite: 74]
df_test.to_csv('company_sales_test.csv', index=False)
print("Test dataset created successfully!")

Test dataset created successfully!


In [2]:
df_test.head()

,Transaction_ID,Product_Category,Sales_Amount,Customer_Age,Rating
0,1,Electronics,150,25.0,5
1,2,Home,200,34.0,4
2,3,Electronics,155,NaN,3
3,4,Sports,300,45.0,5
4,5,Home,210,23.0,2


In [ ]:
import pandas as pd


def automated_stat_analyzer(df, column_name):
    """
    Company Task: Provide a summary report of a specific data variable.

    Instructions:
    1. Check if the column is numerical or categorical.
    2. For numerical: Calculate Mean, Median, and Standard Deviation.
    3. For categorical: Calculate the Mode.
    4. Return a dictionary with these statistical measures.
    """
    series = df[column_name]

    if pd.api.types.is_numeric_dtype(series):
        mean = series.mean()
        median = series.median()
        return {
            "column": column_name,
            "type": "numerical",
            "mean": mean,
            "median": median,
            "std": series.std(),
            "is_skewed": mean != median,
            "nulls": int(series.isna().sum()),
        }

    # categorical column -> return the Mode instead
    return {
        "column": column_name,
        "type": "categorical",
        "mode": list(series.mode()),
        "unique_values": int(series.nunique()),
    }

## Assignment 2: 
  ### The null_handling_strategy Function


#### Scenario: Incoming user data often has missing values.Students must implement a flexible strategy to handle these "Null Values" to prepare data for Machine Learning.
### Requirements:

* Check for null values in the DataFrame.

* Apply a strategy based on parameters: "drop_rows", "fill_mean", or "fill_median" .

* Ensure the function only fills numerical columns when using mean or median.

In [6]:
def null_handling_strategy(df, strategy="fill_mean"):
    """
    Company Task: Clean a dataset by resolving missing (NaN) values.
    """
    null_counts = df.isna().sum()
    total_nulls = int(null_counts.sum())
    print("Null values per column:")
    print(null_counts)
    print("Total nulls:", total_nulls)

    cleaned = df.copy()

    if strategy == "drop_rows":
        cleaned = cleaned.dropna()  # remove rows with any null value

    elif strategy == "fill_mean":
        # only fill numerical columns so we never invent text data
        for col in cleaned.select_dtypes(include="number").columns:
            cleaned[col] = cleaned[col].fillna(cleaned[col].mean())

    elif strategy == "fill_median":
        for col in cleaned.select_dtypes(include="number").columns:
            cleaned[col] = cleaned[col].fillna(cleaned[col].median())

    else:
        raise ValueError("Unknown strategy. Use one of: 'drop_rows', 'fill_mean', 'fill_median'.")

    return cleaned, dict(null_counts)

In [ ]:
## == Assignment 1 tests ==
# numerical column (Sales_Amount has an outlier, so mean and median differ -> skewed)
report1 = automated_stat_analyzer(df_test, "Sales_Amount")
print(report1)

# categorical column -> returns the Mode instead
print(automated_stat_analyzer(df_test, "Product_Category"))

# numerical column with nulls
print(automated_stat_analyzer(df_test, "Customer_Age"))


## == Assignment 2 tests ==
cleaned_mean, null_report = null_handling_strategy(df_test, "fill_mean")
print("Remaining nulls in Customer_Age after fill_mean:", int(cleaned_mean["Customer_Age"].isna().sum()))

cleaned_median, _ = null_handling_strategy(df_test, "fill_median")
print("Remaining nulls in Customer_Age after fill_median:", int(cleaned_median["Customer_Age"].isna().sum()))

cleaned_drop, _ = null_handling_strategy(df_test, "drop_rows")
print("Rows after drop_rows:", len(cleaned_drop))